# Phase 1 — Raw data loading and basic integrity checks (100k.csv)

In [41]:
import pandas as pd
import numpy as np

## 1. Load + column names

In [42]:
df = pd.read_csv("../data_raw/100k_a.csv", header=None)
df.columns = [
    "user_id",
    "stream_id",
    "streamer_name",
    "start_time",
    "stop_time"
]
df.head()

,user_id,stream_id,streamer_name,start_time,stop_time
0,1,33842865744,mithrain,154,156
1,1,33846768288,alptv,166,169
2,1,33886469056,mithrain,587,588
3,1,33887624992,wtcn,589,591
4,1,33890145056,jrokezftw,591,594


## 2. Duration computation

Start and stop times are provided as integers and represent periods of 10 minutes.

In [43]:
df["duration_intervals"] = df["stop_time"] - df["start_time"]
df["duration_minutes"] = df["duration_intervals"] * 10
df.head()

,user_id,stream_id,streamer_name,start_time,stop_time,duration_intervals,duration_minutes
0,1,33842865744,mithrain,154,156,2,20
1,1,33846768288,alptv,166,169,3,30
2,1,33886469056,mithrain,587,588,1,10
3,1,33887624992,wtcn,589,591,2,20
4,1,33890145056,jrokezftw,591,594,3,30


## 3. Integrity checks

In [44]:
# Temporal ordering: start_time must be strictly smaller than stop_time
(df["start_time"] >= df["stop_time"]).sum()

0

In [45]:
# Checking duration distribution to detect extreme or suspicious values
df["duration_minutes"].describe()

count    3.051733e+06
mean     3.142054e+01
std      4.257966e+01
min      1.000000e+01
25%      1.000000e+01
50%      1.000000e+01
75%      3.000000e+01
max      9.700000e+02
Name: duration_minutes, dtype: float64

In [46]:
# Checking for duplicate rows
df.duplicated().sum()

0

## 4. Fragmentation check

We check how many records exist per (user_id, stream_id). Each pair appears exactly once, so consolidation is not needed.

In [47]:
# Count records per (user_id, stream_id) pair
user_stream_counts = df.groupby(["user_id", "stream_id"]).size()

user_stream_counts.describe()

count    3051733.0
mean           1.0
std            0.0
min            1.0
25%            1.0
50%            1.0
75%            1.0
max            1.0
dtype: float64

In [48]:
# Checking if any pair appears more than once
(user_stream_counts > 1).sum()

0

## 5. Dataset summary

In [49]:
# Overall time range
print("Minimum start_time:", df["start_time"].min())
print("Maximum stop_time:", df["stop_time"].max())

Minimum start_time: 0
Maximum stop_time: 6148


In [50]:
# Count unique users and streams
print("Number of unique users:", df["user_id"].nunique())
print("Number of unique streams:", df["stream_id"].nunique())
print("Number of unique streamers:", df["streamer_name"].nunique())

Number of unique users: 100000
Number of unique streams: 739991
Number of unique streamers: 162625


In [51]:
# Final dataset shape after cleaning
print("Final dataset shape:", df.shape)

# Basic duration statistics
print("\nDuration statistics (minutes):")
print(df["duration_minutes"].describe())

Final dataset shape: (3051733, 7)

Duration statistics (minutes):
count    3.051733e+06
mean     3.142054e+01
std      4.257966e+01
min      1.000000e+01
25%      1.000000e+01
50%      1.000000e+01
75%      3.000000e+01
max      9.700000e+02
Name: duration_minutes, dtype: float64


## 6. Save Phase 1 output

The cleaned dataset is saved as a CSV file to be used as the input for Phase 2 (EDA + labeling).

In [52]:
df.to_csv("../data_processed/gold_100k.csv", index=False)

# Dataset Schema

**user_id**  
Unique (anonymized) identifier of the viewer.

**stream_id**  
Unique identifier of the stream session (a specific broadcast instance).

**streamer_name**  
Identifier of the streamer.  
*Multiple stream_id values may belong to the same streamer.*

**start_time**  
Index of the first 10-minute snapshot in which the user was observed in the stream session.

**stop_time**  
Index of the last 10-minute snapshot in which the user was observed in the stream session.

**duration_intervals**  
Difference between stop_time and start_time (number of 10-minute units).  
*This represents the span between first and last observed presence, not necessarily continuous watch time.*

**duration_minutes**  
Approximate interaction window in minutes (duration_intervals × 10).  
*This is an upper-bound estimate derived from snapshot aggregation.*